# train notebook
Kaggle notebook環境で動かすためのコード

## 手順
1. 実験ディレクトリをそのままをkaggle datasetsとしてアップロード
2. notebook内で上記コードを指定 (`src`の親ディレクトリを EXP_ROOT として指定)

In [ ]:
!pip install --upgrade pip
!pip install -q hydra-core onnxruntime-gpu openvino onnxscript schedulefree wandb

In [ ]:
# ruff: noqa: E402
import os
import sys
from pathlib import Path

EXP_ROOT = Path("/kaggle/input/datasets/sorawww31/XXX/experiments").resolve()
EXP_DIR_NAME = "No!"
EXP_DIR = EXP_ROOT / EXP_DIR_NAME
CONFIG_NAME = "config"
sys.path.append(str(EXP_ROOT))
sys.path.append(str(EXP_DIR))

from dotenv import load_dotenv
from hydra.core.config_store import ConfigStore

from src.config import Config, ExpConfig
from src.runtime import (
    build_cfg,
    build_experiment_name,
    init_debug,
    run_experiment,
)
from utils.env import EnvConfig

load_dotenv()

cs = ConfigStore.instance()
cs.store(name="default", group="env", node=EnvConfig)
cs.store(name="default", group="exp", node=ExpConfig)


def main_notebook(exp_choice="default", overrides=None):
    """Single-process entrypoint (CPU or 1 GPU) for exp021_dist.

    DDP 本番 (2GPU+) からはこの関数を使わず、!torchrun ... launch.py ... を使うこと。
    """
    overrides = list(overrides or [])
    cfg = build_cfg(
        exp_choice=exp_choice,
        overrides=overrides,
        config_dir=str(EXP_DIR),
        config_name=CONFIG_NAME,
    )
    init_debug(cfg)
    exp_name = build_experiment_name(
        experiment_dir_name=EXP_DIR_NAME,
        exp_choice=exp_choice,
        config_name=cfg.exp.name,
    )
    run_experiment(cfg, exp_name, overrides=overrides)


In [ ]:
# CPU / 1GPU 検証用: 単一プロセスで実行
main_notebook(
    exp_choice="perch_sed_10s",
    overrides=[
        "env.output_dir=/kaggle/working/outputs",
        "env.input_dir=/kaggle/input",
    ],
)
